In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../dataset/teams.csv")

awards_players = pd.read_csv("../dataset/awards_players.csv")

players_teams = pd.read_csv("../dataset/players_teams.csv")

coaches = pd.read_csv("../dataset/coaches.csv")

df.head()

In [ ]:
# Calculate the number of player awards per team each year
filtered_awards = awards_players[awards_players['award'] != "Coach of the Year"]

award_counts = filtered_awards.groupby(['playerID', 'year']).size().reset_index(name='award_count')

player_team_awards = pd.merge(award_counts, players_teams, on=['playerID', 'year'], how='left')

team_awards = player_team_awards.groupby(['tmID', 'year'])['award_count'].sum().reset_index()

df = pd.merge(df, team_awards[['tmID', 'year', 'award_count']], on=['tmID', 'year'], how='left')

df['award_count'] = df['award_count'].fillna(0).astype(int)

In [ ]:
# Measures how much a team outscores its opponents per game
df["net_rating"] = (df["o_pts"] - df["d_pts"]) / df["GP"]

df["off_eff"] = df["o_pts"] / df["o_fga"]   # points per field goal attempt
df["def_eff"] = df["d_pts"] / df["d_fga"]   # points allowed per opponent FGA

# Measures how effectively the team scores compared to what they allow
df["eff_diff"] = df["off_eff"] - df["def_eff"]

# Calculate the Home vs Away Win Ratio (small gap means consistency)
df["home_win_pct"] = df["homeW"] / (df["homeW"] + df["homeL"])
df["away_win_pct"] = df["awayW"] / (df["awayW"] + df["awayL"])
df["home_away_balance"] = df["away_win_pct"] - df["home_win_pct"]

# Calculate the difference between the rebounds gained by the team and the ones gained by the opponents
df["reb_dif"] = df["o_reb"] - df["d_reb"]

# Calculate the difference between the turnovers of the opponents and the turnovers of the team
df["to_dif"] = df["d_to"] - df["o_to"]

# Calculate the total rebound difference between the team and its opponents
df["tm_reb_dif"] = df["tmTRB"] - df["opptmTRB"]

# Calculate the offensive rebound difference (extra scoring opportunities created by the team)
df["off_reb_dif"] = df["tmORB"] - df["opptmDRB"]

# Calculate the defensive rebound difference (how well the team prevents second-chance points)
df["def_reb_dif"] = df["tmDRB"] - df["opptmORB"]

# Calculate the percentage of all total rebounds captured by the team
df["reb_pct"] = df["tmTRB"] / (df["tmTRB"] + df["opptmTRB"])

# Calculate the percentage of offensive rebounds captured by the team (offensive efficiency)
df["off_reb_pct"] = df["tmORB"] / (df["tmORB"] + df["opptmDRB"])

# Calculate the percentage of defensive rebounds captured by the team (defensive efficiency)
df["def_reb_pct"] = df["tmDRB"] / (df["tmDRB"] + df["opptmORB"])

# Calculate the percentage of conference wins per team
df["conf_win_pct"] = df["confW"] / (df["confW"] + df["confL"])

In [ ]:
# Calculate the number of years that a coach has been with a team, for each specific year
coaches_sorted = coaches.sort_values(['tmID', 'coachID', 'year'])

coaches_sorted['coach_tenure_year'] = (
    coaches_sorted.groupby(['tmID', 'coachID']).cumcount() + 1
)

tenure_by_year = coaches_sorted[['tmID', 'year', 'coach_tenure_year']]

df = df.merge(tenure_by_year, on=['tmID', 'year'], how='left')

df['coach_tenure_year'] = df['coach_tenure_year'].fillna(0).astype(int)

In [ ]:
# Calculate the number of coach awards per team each year
coach_awards = awards_players[awards_players['award'] == 'Coach of the Year']

coach_awards_count = coach_awards.groupby(['playerID', 'year']).size().reset_index(name='coach_awards')

coach_awards_count = pd.merge(coach_awards_count, coaches, left_on=['playerID', 'year'], right_on=['coachID', 'year'], how='left')

df = pd.merge(df, coach_awards_count[['tmID', 'year', 'coach_awards']], on=['tmID', 'year'], how='left')

df['coach_awards'] = df['coach_awards'].fillna(0).astype(int)

In [ ]:
# Calculate win percentage
df["win_pct"] = df["won"] / (df["won"] + df["lost"])

# Calculate offensive field goal percentage
df["fg_pct"] = df["o_fgm"] / (df["o_fgm"] + df["o_fga"])

# Calculate 3 points made percentage
df["3p_pct"] = df["o_3pm"] / (df["o_3pm"] + df["o_3pa"])

# Calculate point average (made points vs allowed points)
df["pa_pct"] = df["o_pts"] / (df["o_pts"] + df["d_pts"])

#
df["ft_pct"] = df["o_ftm"] / (df["o_ftm"] + df["o_fta"])

# Ideas for more features: Cumulative awards (not sure about this one), More offensive and defensive statistics (like blocks and steals)

feature_cols = ["win_pct", "fg_pct", "3p_pct", "pa_pct", "ft_pct", "award_count", "net_rating", "off_eff", 
                "def_eff", "eff_diff", "home_win_pct", "away_win_pct", "home_away_balance",
                "reb_dif", "to_dif", "tm_reb_dif", "off_reb_dif", "def_reb_dif", "reb_pct", 
                "off_reb_pct", "def_reb_pct", "conf_win_pct", "coach_tenure_year", "coach_awards"]

df[feature_cols].head()

print(df.head())

In [ ]:
CORR_THRESHOLD = 0.2

corr = df[["rank"] + feature_cols].corr()

# Filter out all features with low correlation given the correlation threshold
strong_corr = corr["rank"][corr["rank"].abs() >= CORR_THRESHOLD]

strong_corr.sort_values()

In [ ]:
# Set style
sns.set(style="whitegrid", context="notebook")

for col in strong_corr.index:
    if col == "rank":
        continue
    
    plt.figure(figsize=(6, 4))
    sns.regplot(x=col, y="rank", data=df, scatter_kws={"alpha":0.6})
    plt.title(f"Rank vs {col}")
    plt.xlabel(col)
    plt.ylabel("Rank (1 = best)")
    plt.show()

In [ ]:
# Train model
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# Dataframe of predictors
x = df[feature_cols]

# Target variable
y = df["rank"]

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

model = LinearRegression()

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
print(f"Mean squared error: {mse:.4f}")

rmse = mse ** 0.5
print(f"Root mean squared error: {rmse:.4f}")